# LLM 09: Context Windows & Long Context

Hands-on:
1. Measure token usage of real prompts with tiktoken
2. Build a needle-in-a-haystack recall test
3. Compare map-reduce vs refine summarization
4. Simulate lost-in-the-middle by placing facts at different positions
5. Exercises: cost/latency tradeoff, long-context eval harness

Deps: `pip install tiktoken anthropic openai`

## 1. Accounting: How Much Context Are You Using?

In [ ]:
import tiktoken
enc = tiktoken.encoding_for_model('gpt-4o')

def tokens(text):
    return len(enc.encode(text))

components = {
    'system_prompt':         'You are a helpful assistant. ' * 40,
    'tool_schemas':          '{"name": "search", "description": "..."}' * 10,
    'chat_history (10 turns)': 'Hello how are you today? ' * 100,
    'retrieved_docs':        'Lorem ipsum dolor sit amet... ' * 400,
    'user_turn':             'What is the capital of France?',
    'expected_output':       'Paris, as stated in the Treaty of... ' * 30,
}

total = 0
for name, text in components.items():
    t = tokens(text)
    total += t
    print(f'{name:<28} {t:>6} tok ({t*0.0001:.4f} USD @ $0.10/M input)')
print(f'{"TOTAL":<28} {total:>6} tok')
print(f'\nFraction of 128K context window: {total/128000:.1%}')

## 2. Needle in a Haystack: Recall Test

Hide a fact at different positions in a long document; ask the model to retrieve it.

In [ ]:
import os

def call_llm(system, user):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(
            model='claude-sonnet-4-6', max_tokens=100, temperature=0,
            system=system, messages=[{'role':'user','content':user}])
        return r.content[0].text
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(
            model='gpt-4o-mini', temperature=0,
            messages=[{'role':'system','content':system},{'role':'user','content':user}])
        return r.choices[0].message.content
    return '[skipped - no API key]'

filler_sentence = 'The weather report was unremarkable. '
needle = 'The password to the vault is OCTOPUS-42.'

def build_haystack(needle_position_frac: float, total_chars: int = 20000) -> str:
    n = total_chars // len(filler_sentence)
    pre  = int(n * needle_position_frac)
    post = n - pre
    return filler_sentence * pre + needle + ' ' + filler_sentence * post

for pos in [0.0, 0.25, 0.5, 0.75, 1.0]:
    haystack = build_haystack(pos)
    ans = call_llm(
        'Answer from the provided document only.',
        f'{haystack}\n\nQ: What is the password to the vault? Answer only with the password.',
    )
    tokens_used = tokens(haystack)
    print(f'pos={pos:.2f}  tok={tokens_used:>5}  answer: {ans.strip()[:80]}')

## 3. Map-Reduce Summarization (fits in short context)

In [ ]:
long_doc = '''
Chapter 1. In 1920, the small town of Willowbrook elected its first mayor, Elinor Hayes. ...
Chapter 2. By 1935, the town had grown to 4,000 residents and built its first library. ...
Chapter 3. In 1972, Willowbrook's textile mill closed, triggering two decades of decline. ...
Chapter 4. The 2008 revitalization project, led by mayor Denise Park, introduced ... 
''' * 3

chunks = long_doc.strip().split('Chapter ')
chunks = ['Chapter ' + c for c in chunks if c.strip()]

partial_summaries = []
for i, chunk in enumerate(chunks):
    s = call_llm('Summarize in 1 sentence.', chunk)
    partial_summaries.append(s)
    print(f'chunk {i}: {s.strip()[:120]}')

combined = call_llm(
    'Combine these per-chapter summaries into a 3-sentence document summary.',
    '\n'.join(partial_summaries))
print('\nCombined summary:\n', combined)

## 4. Refine Summarization (carries state)

In [ ]:
running = ''
for i, chunk in enumerate(chunks):
    running = call_llm(
        'Refine the running summary to include the new chapter. Keep it under 3 sentences.',
        f'RUNNING SUMMARY:\n{running}\n\nNEW CHAPTER:\n{chunk}',
    )
    print(f'after chunk {i}: {running.strip()[:160]}')

## 5. Exercise: Cost vs Quality Tradeoff

Build a small dataset of 10 (document, question, expected_answer) triples.

For each, run three strategies and compute (accuracy, avg tokens, avg latency, avg cost):

1. **Whole doc in context.**
2. **Summary in context** (pre-computed via map-reduce).
3. **Retrieval of top-3 chunks** (see `09-rag/`).

Expect retrieval to win on cost and often tie or beat on quality. This is the whole argument for RAG.

## 6. Exercise: Build a Needle Grid Benchmark

Vary *both* the context length (25%, 50%, 75%, 100% of advertised) and the needle position (0.0, 0.25, 0.5, 0.75, 1.0). Plot the recall as a heatmap.

Real frontier models show interesting patterns: strong at start/end, dipping in the middle, degrading as context approaches the advertised max. This is **the** test to run before trusting a model's advertised context length.

## 7. Takeaways

- **Context is a shared budget** across system, history, retrieval, and generation.
- **Effective context ≤ advertised context.** Measure on your task.
- **Lost-in-the-middle is real.** Position matters.
- **RAG is usually cheaper and often better** than long-context for chat-with-your-docs.
- **Prompt caching changes the cost calculus** for repeated long prefixes.

Next: **10 — Streaming & Incremental Rendering**, where we go from batch responses to real-time UX.